# AWS Bedrock POC - Transaction Categorisation

Using Amazon Nova models (no use case form required).

## 1. Configuration

In [ ]:
import boto3
import json
from typing import Optional

AWS_REGION = "ap-southeast-2"

# Models available in Sydney
MODELS = {
    "nova_lite": "amazon.nova-lite-v1:0",
    "nova_pro": "amazon.nova-pro-v1:0",
    "nova_micro": "amazon.nova-micro-v1:0",
    "claude_35_sonnet_v2": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "claude_3_haiku": "anthropic.claude-3-haiku-20240307-v1:0",
    "claude_3_sonnet": "anthropic.claude-3-sonnet-20240229-v1:0",
}

DEFAULT_MODEL = MODELS["claude_3_haiku"]

## 2. Bedrock Client (Nova Format)

In [ ]:
class BedrockClient:
    """Wrapper for Bedrock inference - supports Anthropic and Amazon models."""
    
    def __init__(self, region: str = AWS_REGION, model_id: str = DEFAULT_MODEL):
        self.client = boto3.client("bedrock-runtime", region_name=region)
        self.model_id = model_id
        self.provider = "anthropic" if model_id.startswith("anthropic.") else "amazon"
    
    def _build_body(self, prompt: str, system: Optional[str], 
                    max_tokens: int, temperature: float) -> dict:
        if self.provider == "anthropic":
            body = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}]
            }
            if system:
                body["system"] = system
        else:
            body = {
                "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
                "messages": [{"role": "user", "content": [{"text": prompt}]}]
            }
            if system:
                body["system"] = [{"text": system}]
        return body
    
    def _parse_response(self, result: dict) -> dict:
        if self.provider == "anthropic":
            content = result["content"][0]["text"]
            stop = result.get("stop_reason")
        else:
            content = result["output"]["message"]["content"][0]["text"]
            stop = result.get("stopReason")
        
        return {
            "content": content,
            "usage": result.get("usage", {}),
            "model": self.model_id,
            "stop_reason": stop
        }
        
    def invoke(self, prompt: str, system: Optional[str] = None,
               max_tokens: int = 1024, temperature: float = 0.0) -> dict:
        response = self.client.invoke_model(
            modelId=self.model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(self._build_body(prompt, system, max_tokens, temperature))
        )
        return self._parse_response(json.loads(response["body"].read()))

## 3. Test Connection

In [ ]:
bedrock = BedrockClient()

bedrock = BedrockClient(model_id=MODELS["claude_35_sonnet_v2"])
response = bedrock.invoke("Say 'Bedrock connection successful!' and nothing else.")
print(f"Response: {response['content']}")
print(f"Tokens used: {response['usage']}")

## 4. Transaction Categorisation Setup

In [ ]:
CATEGORY_TAXONOMY = """
Categories:
1. GROCERIES - Supermarkets, food stores, grocery delivery
2. DINING - Restaurants, cafes, fast food, food delivery apps
3. TRANSPORT - Fuel, public transport, rideshare, parking
4. UTILITIES - Electricity, gas, water, internet, phone
5. ENTERTAINMENT - Streaming, movies, games, concerts
6. SHOPPING - Retail, clothing, electronics, online marketplaces
7. HEALTH - Pharmacy, medical, fitness, wellness
8. TRAVEL - Hotels, flights, car rental, travel agencies
9. FINANCIAL - Bank fees, insurance, investments
10. OTHER - Uncategorised transactions
"""

BUSINESS_RULES = """
Rules:
- "UBER" or "LYFT" → TRANSPORT (not DINING even if Uber Eats)
- Amounts < $20 at restaurants → likely DINING
- "PRIME" or "AMZN" → SHOPPING unless clearly a streaming charge
- Recurring monthly charges → likely UTILITIES or ENTERTAINMENT
"""

FEW_SHOT_EXAMPLES = """
Examples:
Transaction: "WOOLWORTHS 1234 SYDNEY" $85.50 → GROCERIES
Transaction: "UBER *TRIP HELP.UBER.COM" $23.40 → TRANSPORT
Transaction: "NETFLIX.COM" $22.99 → ENTERTAINMENT
Transaction: "BP SERVICE STATION" $65.00 → TRANSPORT
Transaction: "MENULOG*THAI PALACE" $42.00 → DINING
"""

In [ ]:
def build_categorisation_prompt(transaction_desc: str, amount: float,
                                  taxonomy: str = CATEGORY_TAXONOMY,
                                  rules: str = BUSINESS_RULES,
                                  examples: str = FEW_SHOT_EXAMPLES) -> tuple:
    system_prompt = f"""You are a financial transaction categorisation system.
Your task is to categorise bank transactions into the correct category.

{taxonomy}

{rules}

{examples}

Respond with ONLY the category name. Be consistent and precise."""

    user_prompt = f'Categorise this transaction: "{transaction_desc}" Amount: ${amount:.2f}'
    return system_prompt, user_prompt


def categorise_transaction(description: str, amount: float,
                           client: BedrockClient = None, **kwargs) -> dict:
    if client is None:
        client = BedrockClient()
    
    system, user = build_categorisation_prompt(description, amount, **kwargs)
    response = client.invoke(prompt=user, system=system, max_tokens=100, temperature=0.0)
    
    return {
        "transaction": description,
        "amount": amount,
        "category": response["content"].strip(),
        "tokens": response["usage"],
        "model": response["model"]
    }

## 5. Test Categorisation

In [ ]:
test_transactions = [
    ("COLES SUPERMARKET MELBOURNE", 125.50),
    ("UBER *TRIP HELP.UBER.COM", 18.90),
    ("SPOTIFY SYDNEY", 12.99),
    ("SHELL SERVICE STATION", 78.00),
    ("AMAZON PRIME*1234", 9.99),
    ("DR SMITH MEDICAL CTR", 85.00),
]

print("Transaction Categorisation Results")
print("=" * 50)

total_tokens = {"inputTokens": 0, "outputTokens": 0}

for desc, amount in test_transactions:
    result = categorise_transaction(desc, amount, client=bedrock)
    print(f"\n{desc} (${amount})")
    print(f"  → {result['category']}")
    print(f"  Tokens: {result['tokens']}")
    
    total_tokens["inputTokens"] += result["tokens"].get("inputTokens", 0)
    total_tokens["outputTokens"] += result["tokens"].get("outputTokens", 0)

print(f"\n{'=' * 50}")
print(f"Total tokens used: {total_tokens}")

## 6. Cost Estimation

In [ ]:
# Nova pricing (USD per 1K tokens)
PRICING = {
    "amazon.nova-lite-v1:0": {"input": 0.00006, "output": 0.00024},
    "amazon.nova-pro-v1:0": {"input": 0.0008, "output": 0.0032},
    "amazon.nova-micro-v1:0": {"input": 0.000035, "output": 0.00014},
}

def estimate_cost(input_tokens: int, output_tokens: int, model_id: str) -> dict:
    prices = PRICING.get(model_id, PRICING["amazon.nova-lite-v1:0"])
    input_cost = (input_tokens / 1000) * prices["input"]
    output_cost = (output_tokens / 1000) * prices["output"]
    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
        "cost_per_1k_transactions": (input_cost + output_cost) * 1000 / len(test_transactions)
    }

cost = estimate_cost(total_tokens["inputTokens"], total_tokens["outputTokens"], DEFAULT_MODEL)
print(f"Cost for {len(test_transactions)} transactions: ${cost['total_cost']:.6f}")
print(f"Projected cost per 1,000 transactions: ${cost['cost_per_1k_transactions']:.4f}")

## 7. Next Steps

1. Once Anthropic use case form is approved, switch to Claude models
2. Compare Nova vs Claude accuracy on same test set
3. Load production data for real accuracy benchmarking
4. Experiment with different prompts and taxonomy

In [ ]:
bedrock = BedrockClient(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0")
response = bedrock.invoke("Say 'hello' and nothing else.")
print(response["content"])


In [ ]:
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

# Claude format (different from Nova)
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 100,
    "temperature": 0.0,
    "messages": [
        {"role": "user", "content": "Say 'Bedrock connection successful!' and nothing else."}
    ]
}

response = client.invoke_model(
    modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps(body)
)

result = json.loads(response["body"].read())
print(result["content"][0]["text"])

In [ ]:
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

# Claude format (different from Nova)
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 100,
    "temperature": 0.0,
    "messages": [
        {"role": "user", "content": "Please tell me a joke related to mothers"}
    ]
}

response = client.invoke_model(
    modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",  # Cheapest/fastest in Sydney
    contentType="application/json",
    accept="application/json",
    body=json.dumps(body)
)

result = json.loads(response["body"].read())
print(result["content"][0]["text"])